# PennyLane QNodes and analytic measurements

Bind one quantum function to default.qubit and MettleQ, then compare state, probabilities, expectation, and variance.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    total_variation_distance,
)

In [ ]:
def make_qnode(device):
    @qml.qnode(device)
    def circuit(theta):
        qml.Hadamard(0)
        qml.CNOT(wires=[0, 1])
        qml.Rot(theta, -0.21, 0.13, wires=1)
        return qml.state(), qml.probs(wires=[1, 0]), qml.expval(qml.X(0) @ qml.Z(1)), qml.var(qml.Z(0))
    return circuit

reference_device = qml.device("default.qubit", wires=2)
reference_qnode = make_qnode(reference_device)
reference, reference_ms, _ = benchmark(lambda: reference_qnode(0.31))
mettleq_device = MettleQDevice(wires=2, method="statevector", device="cpu")
mettleq_qnode = make_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: mettleq_qnode(0.31))
errors = [max_abs_error(left, right) for left, right in zip(reference, candidate)]
method, device = pennylane_selection(mettleq_device)
tutorial_result = emit_result(
    notebook="pennylane/01_qnodes_and_measurements.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="all analytic measurements atol=2e-6",
    passed=max(errors) <= 2e-6,
    exact_match=all(np.array_equal(np.asarray(left), np.asarray(right)) for left, right in zip(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"per_measurement_max_errors": errors},
)